In [42]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder.appName("NarrorAndWideTransformation")
    # .config("spark.executor.memory", "1g")
    .getOrCreate()
)

# 2. Create 10 rows with bad column names & realistic edge cases (nulls, negative qty)
raw_data = [
    (1, "Laptop", "Electronics", 1200.0, 1, "Completed"),
    (2, "Desk Chair", "Furniture", 250.0, 2, "Completed"),
    (3, "Wireless Mouse", "Electronics", 35.0, 4, "Completed"),
    (4, "Monitor", "Electronics", None, 1, "Completed"),  # Null price
    (5, "Standing Desk", "Furniture", 550.0, 1, "Completed"),
    (6, "Webcam", "Electronics", 85.0, -2, "Completed"),  # Invalid negative quantity
    (7, "Notebook Pack", "Stationery", 12.0, 10, "Cancelled"),  # Cancelled order
    (8, "Desk Lamp", "Furniture", 45.0, 3, "Completed"),
    (9, "USB-C Hub", "Electronics", 60.0, 2, "Completed"),
    (10, "Pen Set", "Stationery", 15.0, 5, "Completed"),
]

# Intentional bad column names (spaces, capital letters, symbols)
bad_columns = [
    "Order ID",
    "Product Name",
    "Category",
    "Unit Price ($)",
    "Qty",
    "Order_Status",
]

df = spark.createDataFrame(raw_data, schema=bad_columns)

# df.show(20)

### NARROW TRANSFORMATION

Narrow transformations are functions that doesn't require moving data accros the executors, basically they don't trigger Data Exchange or Shuffle 
- `.filter()`: Filtering a row based on a condition. [Documentation Here](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.filter.html#pyspark.sql.DataFrame.filter)
- `.select()`: Selecting a specific column. [Documenation Here](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.select.html)
- `.withColumn()`: Add a new column. [Documentation Here](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.withColumn.html)
- `.withColumnRenamed()`: Rename a column. [Documentation Here](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.withColumnRenamed.html)
- `.drop()`: Drop a specific columns. [Documentation Here](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.drop.html)

In [40]:
#filter Example

df.filter(df["Unit Price ($)"] > 250.0).show()

[Stage 0:>                                                          (0 + 1) / 1]

+--------+-------------+-----------+--------------+---+------------+
|Order ID| Product Name|   Category|Unit Price ($)|Qty|Order_Status|
+--------+-------------+-----------+--------------+---+------------+
|       1|       Laptop|Electronics|        1200.0|  1|   Completed|
|       5|Standing Desk|  Furniture|         550.0|  1|   Completed|
+--------+-------------+-----------+--------------+---+------------+



In [13]:
# select Example

df.select('Category', 'Order_Status').show(6)

+-----------+------------+
|   Category|Order_Status|
+-----------+------------+
|Electronics|   Completed|
|  Furniture|   Completed|
|Electronics|   Completed|
|Electronics|   Completed|
|  Furniture|   Completed|
|Electronics|   Completed|
+-----------+------------+
only showing top 6 rows


In [17]:
# .withColumn() Example

df.withColumn('after_discount', df['Unit Price ($)'] - 5).show(2)

+--------+------------+-----------+--------------+---+------------+--------------+
|Order ID|Product Name|   Category|Unit Price ($)|Qty|Order_Status|after_discount|
+--------+------------+-----------+--------------+---+------------+--------------+
|       1|      Laptop|Electronics|        1200.0|  1|   Completed|        1195.0|
|       2|  Desk Chair|  Furniture|         250.0|  2|   Completed|         245.0|
+--------+------------+-----------+--------------+---+------------+--------------+
only showing top 2 rows


In [18]:
# .withColumnRenamed() Example

df.withColumnRenamed("Unit Price ($)", "unit_price").show()

+--------+--------------+-----------+----------+---+------------+
|Order ID|  Product Name|   Category|unit_price|Qty|Order_Status|
+--------+--------------+-----------+----------+---+------------+
|       1|        Laptop|Electronics|    1200.0|  1|   Completed|
|       2|    Desk Chair|  Furniture|     250.0|  2|   Completed|
|       3|Wireless Mouse|Electronics|      35.0|  4|   Completed|
|       4|       Monitor|Electronics|      NULL|  1|   Completed|
|       5| Standing Desk|  Furniture|     550.0|  1|   Completed|
|       6|        Webcam|Electronics|      85.0| -2|   Completed|
|       7| Notebook Pack| Stationery|      12.0| 10|   Cancelled|
|       8|     Desk Lamp|  Furniture|      45.0|  3|   Completed|
|       9|     USB-C Hub|Electronics|      60.0|  2|   Completed|
|      10|       Pen Set| Stationery|      15.0|  5|   Completed|
+--------+--------------+-----------+----------+---+------------+



In [19]:
# Rename multiple columns with withColumnsRenamed
df.withColumnsRenamed(
    {
        "Order ID": "order_id", 
        "Product Name": "product_name",
        "Unit Price ($)": "unit_price"
    }
).show()

+--------+--------------+-----------+----------+---+------------+
|order_id|  product_name|   Category|unit_price|Qty|Order_Status|
+--------+--------------+-----------+----------+---+------------+
|       1|        Laptop|Electronics|    1200.0|  1|   Completed|
|       2|    Desk Chair|  Furniture|     250.0|  2|   Completed|
|       3|Wireless Mouse|Electronics|      35.0|  4|   Completed|
|       4|       Monitor|Electronics|      NULL|  1|   Completed|
|       5| Standing Desk|  Furniture|     550.0|  1|   Completed|
|       6|        Webcam|Electronics|      85.0| -2|   Completed|
|       7| Notebook Pack| Stationery|      12.0| 10|   Cancelled|
|       8|     Desk Lamp|  Furniture|      45.0|  3|   Completed|
|       9|     USB-C Hub|Electronics|      60.0|  2|   Completed|
|      10|       Pen Set| Stationery|      15.0|  5|   Completed|
+--------+--------------+-----------+----------+---+------------+



### WIDE TRANSFORMATION

Basically pyspark function that triggers Data exchange or Shuffling of data from one Executor to the Other.

- `groupBy().agg()`: Filtering a row based on a condition. [Documentation Here](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.groupBy.html)
- `.join()`: Selecting a specific column. [Documenation Here](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html)
- `.orderBy() / .sort()`: Add a new column. [Documentation Here](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.orderBy.html)

In [43]:
df.groupBy("Category").agg(F.sum("Unit Price ($)")).show()

[Stage 0:>                                                          (0 + 2) / 2]

+-----------+-------------------+
|   Category|sum(Unit Price ($))|
+-----------+-------------------+
|Electronics|             1380.0|
|  Furniture|              845.0|
| Stationery|               27.0|
+-----------+-------------------+



In [29]:
# OrderBy Example

df.orderBy(df['Unit Price ($)'].desc()).show()

+--------+--------------+-----------+--------------+---+------------+
|Order ID|  Product Name|   Category|Unit Price ($)|Qty|Order_Status|
+--------+--------------+-----------+--------------+---+------------+
|       1|        Laptop|Electronics|        1200.0|  1|   Completed|
|       5| Standing Desk|  Furniture|         550.0|  1|   Completed|
|       2|    Desk Chair|  Furniture|         250.0|  2|   Completed|
|       6|        Webcam|Electronics|          85.0| -2|   Completed|
|       9|     USB-C Hub|Electronics|          60.0|  2|   Completed|
|       8|     Desk Lamp|  Furniture|          45.0|  3|   Completed|
|       3|Wireless Mouse|Electronics|          35.0|  4|   Completed|
|      10|       Pen Set| Stationery|          15.0|  5|   Completed|
|       7| Notebook Pack| Stationery|          12.0| 10|   Cancelled|
|       4|       Monitor|Electronics|          NULL|  1|   Completed|
+--------+--------------+-----------+--------------+---+------------+



In [30]:
# Sort 

df.sort("Order ID", ascending=True).show()

+--------+--------------+-----------+--------------+---+------------+
|Order ID|  Product Name|   Category|Unit Price ($)|Qty|Order_Status|
+--------+--------------+-----------+--------------+---+------------+
|       1|        Laptop|Electronics|        1200.0|  1|   Completed|
|       2|    Desk Chair|  Furniture|         250.0|  2|   Completed|
|       3|Wireless Mouse|Electronics|          35.0|  4|   Completed|
|       4|       Monitor|Electronics|          NULL|  1|   Completed|
|       5| Standing Desk|  Furniture|         550.0|  1|   Completed|
|       6|        Webcam|Electronics|          85.0| -2|   Completed|
|       7| Notebook Pack| Stationery|          12.0| 10|   Cancelled|
|       8|     Desk Lamp|  Furniture|          45.0|  3|   Completed|
|       9|     USB-C Hub|Electronics|          60.0|  2|   Completed|
|      10|       Pen Set| Stationery|          15.0|  5|   Completed|
+--------+--------------+-----------+--------------+---+------------+



In [44]:
spark.stop()